# Food Shelf List Reconciliation

This short notebook examines differences between the available lists of food shelves.

For in-depth cleaning and analysis, see `food_shelf_eda.ipynb`.

In [1]:
import pandas as pd

## Option 1: Simple Scraper

`mn_food_shelf_list.csv` from `scrape_food_shelf_list.py`.

This scraper pulls the `name` and `address` all 552 food shelves from [the Food Group's Find Help Map](https://www.hungersolutions.org/find-help/?fwp_categories=food-shelves).

In [2]:
# Option 1
shelf1 = pd.read_csv("data/mn_food_shelf_list.csv")
len(shelf1)

552

In [ ]:
# One example row:
shelf1.sample(1, random_state=42)

,name,address
547,Worthington Christian Church Food Shelf( miles...,"1501 N Douglas Ave, Worthington, MN 56187"


Because `scrape_food_shelf_list.py` only pulls `address`, we need to programmatically attempt to parse and locate each address so we can identify the correct county. These attempts were unsuccessful.

## Option 2: Scraper + Geocoder

`food_shelves_scrapedgeocoded.pkl` from `scrape_and_geocode.py`.

This more robust script uses the latitude and longitude and [the Census Geocoder](https://geocoding.geo.census.gov/geocoder/) to create a table with `name`, `address`, `lat`, `lng`, and `county`.

In [ ]:
# Option 2:

shelf2 = pd.read_pickle("data/food_shelves_scrapedgeocoded_2026.pkl")
len(shelf2)

549

In [ ]:
# One example row:
shelf2.sample(1, random_state=42)

,name,address,lat,lng,county
195,Hastings Family Service,"301 East 2nd Street, Hastings, MN 55033",44.74465,-92.849634,Dakota County


In [ ]:
# Checking for missing or mangled counties:
shelf2.loc[shelf2["county"].isna() | (shelf2["county"].str.len() < 5), "county"].count()

np.int64(0)

The data are clean and usable, although 3 food shelves are missing.

In [ ]:
# Searching for discrepencies between the two scraped lists:
com_shelf = pd.concat([shelf1, shelf2])
com_shelf = com_shelf.drop_duplicates("address", keep=False)
com_shelf.loc[:, ["name", "address", "county"]].sort_values("name")

,name,address,county
66,Century College Food Pantry,Century College,Washington County
66,Century College Food Pantry( miles away),"Century College\n3300 Century Ave N, White Bea...",NaN
94,Comunidades Latinas Unidas En Servicio (CLUES)...,NaN,NaN
101,Deer River Area Food Shelf,Goodall Resource Center,Itasca County
102,Deer River Area Food Shelf( miles away),"Goodall Resource Center\n1049 Comstock Drive, ...",NaN
439,Second Harvest Northland: Koochiching County( ...,"10370 MN-11, Birchdale, MN, USA",NaN
542,Wheaton Community Food Shelf,First Presbyterian Church,Traverse County
543,Wheaton Community Food Shelf( miles away),"First Presbyterian Church\n905 2nd Ave S, Whea...",NaN


Comparing the discrepencies between the two lists, `scrape_and_geocode.py` only grabbed part of the address for three food shelves - Century College Food Pantry, Deer River Area Food Shelf, and Wheaton Community Food Shelf - but correctly geolocated the county using latitude and longitude. 

Two other food shelves are missing from list #2: CLUES & Second Harvest Northland: Koochiching County. We can re-add those food shelves manually.

In [27]:
# Looking for the missing CLUES:
shelf2.loc[shelf2["name"].str.contains("CLUES")]

,name,address,lat,lng,county
80,CLUES Canasta Familiar – Austin,"1900 8th Ave NW, Austin, MN 55912, USA",43.676150,-92.996153,Mower County
94,Comunidades Latinas Unidas En Servicio (CLUES)...,"777 East Lake Street, Minneapolis, MN 55407, USA",44.948053,-93.262999,Hennepin County
549,Comunidades Latinas Unidas En Servicio (CLUES)...,"797 E 7th St, Saint Paul, MN 55106",NaN,NaN,Ramsey County


The Hennepin County CLUES is already added, but the Ramsey County CLUES is missing (this is because the Ramsey County CLUES is missing an address). We add it manually:

In [26]:
# Adding Comunidades Latinas Unidas En Servicio (CLUES)- Saint Paul
shelf2.loc[len(shelf2)] = [
    "Comunidades Latinas Unidas En Servicio (CLUES)- Saint Paul",
    "797 E 7th St, Saint Paul, MN 55106",
    pd.NA,
    pd.NA,
    "Ramsey County",
]

In [ ]:
# Looking for Second Harvest Northland: Koochiching County
shelf2.loc[
    shelf2["name"].str.contains("Second Harvest Northland"),
    ["name", "address", "county"],
].sort_values("county")

,name,address,county
425,Second Harvest Northland: Aitkin County,"106 7th Ave Palisade, MN",Aitkin County
430,Second Harvest Northland: Carlton County,"615 12th Street, Cloquet, MN, USA",Carlton County
426,Second Harvest Northland: Backus Mobile,"210 1st Ave, Backus, MN, USA",Cass County
431,Second Harvest Northland: Cook County,"73 Upper Road, Grand Portage, MN, USA",Cook County
432,Second Harvest Northland: Crow Wing County,"501 W College Dr, Brainerd, MN, USA",Crow Wing County
433,Second Harvest Northland: Douglas County,"11523 S Business 53, Solon Springs, Wisconsin,...",Douglas County
434,Second Harvest Northland: Iron County,"606-607 3rd Avenue North, Hurley, WI, USA",Iron County
424,Second Harvest Northland West Facility,"2222 Cromell Drive, Grand Rapids, MN 55744",Itasca County
427,Second Harvest Northland: Ball Club,"30995 Arctic Rd, Deer River, MN, USA",Itasca County
436,Second Harvest Northland: Itasca County,"53714 County Road 146, Inger, MN, USA",Itasca County


Second Harvest Northland: Koochiching County has an address of 10370 MN-11, Birchdale, MN, USA. In comparison, Second Harvest Northland: Birchdale Mobile has an address of 10370 M-11 Birchdale, MN 56629, which is very similar. It may be the duplicate "MN" caused the problem. The website for [Second Harvest Northland's Mobile Food Pantry Program](https://northernlakesfoodbank.org/find-food/mobile-food-pantry-program/) doesn't make it clear whether or not these are distinct services. I'm going to err towards counting them as unique offerings, and manually add Second Harvest Northland: Koochiching County:

In [31]:
# Adding Second Harvest Northland: Koochiching County
shelf2.loc[len(shelf2)] = [
    "Second Harvest Northland: Koochiching County",
    "10370 MN-11, Birchdale, MN, USA",
    pd.NA,
    pd.NA,
    "Koochiching County",
]

In [15]:
# Show non-Minnesota food shelves:
(
    shelf2.loc[~shelf2["address"].str.contains("MN")]
    .loc[~shelf2["address"].str.contains("Mn")]
    .loc[~shelf2["address"].str.contains("Minnesota")]
    .loc[:, ["name", "address", "county"]]
    .reset_index(drop=True)
)

,name,address,county
0,Century College Food Pantry,Century College,Washington County
1,Deer River Area Food Shelf,Goodall Resource Center,Itasca County
2,Dorothy Day Food Pantry West Fargo,"45 21st Ave W, West Fargo, ND 58078, USA",Cass County
3,Emergency Food Pantry,"1101 4th Ave N, Fargo, ND 58102, USA",Cass County
4,Family Pathways Frederic,"1100 Wisconsin Ave, Frederic, WI 54837, USA",Polk County
5,Family Pathways St. Croix Falls,"2000 U.S. 8, St Croix Falls, WI 54024, USA",Polk County
6,New Life Center,"1902 3rd Ave N, Fargo, ND 58102, USA",Cass County
7,Pierce County Food Pantry,"440 N Maple St, Ellsworth, WI 54011, USA",Pierce County
8,Richland Wilkin Food Pantry,"699 8th Ave S, Wahpeton, ND 58075",Richland County
9,Salvation Army Fargo,"304 Roberts Street, Fargo, ND 58102",Cass County


## Option 3: Older ECLDS data

A list of food shelves downloaded from the Minnesota Early Childhood Longitudinal Data System ([ECLDS](https://eclds.mn.gov/))'s [MN Family Resource Map](https://pub.education.mn.gov/mnfr/) in mid-April 2026. The Resource Map authors state they compiled their list of food shelves from Hunger Solutions, which partnered with The Food Group in March 2024. I estimate this list is from 2024 or 2025.

In [33]:
shelf3 = pd.read_csv("data/eclds_food_shelf_list.csv")
len(shelf3)

526

In [ ]:
# Sample row:
shelf3.sample(1, random_state=42)

,"""Map Location""",ProviderName,ServiceName,ProviderType,Website,LocationName,Address,County,Email,Phone
311,Show on Map,New Creation Baptist Church,Food Shelf,Faith Community,https://newcreationbaptistchurchmn.org,New Creation Baptist Church,"1414 E 48th St, Minneapolis MN 55417",Hennepin,NaN,612-825-6933 x


This dataset is appealing because it's already cleaned, labels county, and provides addition information, such as `ProviderType`.

In [49]:
# Number of Food Shelves:
numshelf = len(shelf3.loc[shelf3["ServiceName"] == "Food Shelf"])
numfooddist = len(shelf3.loc[shelf3["ServiceName"] == "Food Distribution"])
print(f"Num Food Shelves: {numshelf}\nNum Food Distribution: {numfooddist}")

Num Food Shelves: 482
Num Food Distribution: 44


In [ ]:
# Comparing the ECLDS list to the option 2 list:
# First five food shelves for shelf 3:
shelf3.loc[
    shelf3["ServiceName"] == "Food Shelf", ["ProviderName", "Address", "County"]
].head(5)

,ProviderName,Address,County
42,360 Communities,"501 Highway 13 E, Ste 112, Burnsville MN 55337",Dakota
43,360 Communities,"510 Walnut St, Door 8, Farmington MN 55024",Dakota
45,360 Communities,"14521 Cimarron Ave W, Rosemount MN 55068",Dakota
46,Akeley/Nevis Community Food Shelf,"6 Broadway Street East, Akeley MN 56433",Hubbard
47,360 Communities,"16725 Highview Ave, Messiah Lutheran Church, L...",Dakota


In [55]:
# First ten food shelves for list two:
shelf2.loc[:, ["name", "address", "county"]].head(5)

,name,address,county
0,360 Communities Apple Valley Food Shelf,"12650 Johnny Cake Ridge Road, Apple Valley, MN...",Dakota County
1,360 Communities Burnsville Food Shelf,"501 East Highway 13 Suite 112, Burnsville, MN ...",Dakota County
2,360 Communities Farmington Food Shelf,"510 Walnut Street, Farmington, MN 55024",Dakota County
3,360 Communities Messiah Community Food Shelf,"16725 Highview Ave, Lakeville, MN 55044",Dakota County
4,360 Communities Rosemount Food Shelf,"14521 Cimarron Ave W, Rosemount, MN 55068",Dakota County


The lists are similar, but Option 2 - the scraped and geocoded list - has two advantages: it has more food shelves (551 versus 482), and it is more reliably up-to-date. This project uses Option 2 for analysis.